--------------------------------------------------
# **Create Czech text dataset for training**
-------------------------------------------------

In [1]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


## ***Imports***

In [2]:
import os, sys
import zipfile
import subprocess
import io
import shutil
import json
import polars as pl
import numpy as np
import pyarrow as pa
import pickle

import pyarrow.parquet as pq
import pyarrow.dataset as ds

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    HfApi
)


from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm

## ***Hugging Face Login***

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
user_email = user_secrets.get_secret("user_email")
user_name = user_secrets.get_secret("user_name")

login(token=hf_token)

### ***Data files***

1. ***CZLC/cermat_czech_mc***
   
   The **Cermat Czech MultiChoice** dataset was collected from assignments official CERMAT website. The dataset was collected from three tiers of assignments: 6 year, 9 year primary school test and final high school tests (so-called maturita). The assignments were semi-manually extracted from official PDFs available at [CERMAT's website](https://maturita.cermat.cz/menu/testy-a-zadani-z-predchozich-obdobi/matematika/testy-a-zadani-matematika).

    **Collection Date Range:** years 2016-2023

    ***Licensing and Credits***

    The majority of collection work was done by our student co-worker Jan Kapsa. Members of CZLC do not own the test assignments, neither are responsible for their contents.

    **Source:** https://huggingface.co/datasets/CZLC/cermat_czech_mc
 
2. ***hynky/czech-justice-summ-alpaca-long***

    **Source:** https://huggingface.co/datasets/hynky/czech-justice-summ-alpaca-long

3. ***hynky/czech_news_dataset_v2***

    Dataset containing the news articles from major online news outlets collected from 2000-2022.
    Follow-up paper https://arxiv.org/abs/2307.10666 (v1 of the dataset)

    Changes from v1:
        - Better contribution of novinky.cz in later stages
        - More articles, as a mistake in filtering was fixed.

   Collection was done using CmonCrawl.

   The dataset should be used for Research only purposes as I don't have rights for articles itself.

   **Source:** https://huggingface.co/datasets/hynky/czech_news_dataset_v2

4. ***davidadamczyk/czechbench_agree***

   This is an adapted and filtered test subset from the original Czech grammar agreement dataset

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_agree

6. ***davidadamczyk/czechbench_czech_news***

   This is a simplified and subsampled test subset from the original [czech_news_dataset_v2](https://huggingface.co/datasets/hynky/czech_news_dataset_v2).

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_czech_news

8. ***CIIRC-NLP/czech_news_simple-cs***

   **Source:** https://huggingface.co/datasets/CIIRC-NLP/czech_news_simple-cs

   This is a simplified and subsampled test subset from the original czech_news_dataset_v2.

   Only 5 basic news categories are considered:

   - Foreign
   - Local
   - Sports
   - Economy

   The test set includes 200 examples per category, 1000 examples in total. Apart from the category label, each example also contains the article's headline, brief summary, full textual content, optional keywords, original category specification, and URL.

   This dataset was created for use within the [Czech-Bench](https://gitlab.com/jirkoada/czech-bench) evaluation framework. 

In [4]:
ds01 = load_dataset("CZLC/cermat_czech_mc")
ds02 = load_dataset("hynky/czech-justice-summ-alpaca-long")
ds03 = load_dataset("hynky/czech_news_dataset_v2")
ds04 = load_dataset("davidadamczyk/czechbench_agree")
ds05 = load_dataset("davidadamczyk/czechbench_czech_news")
ds06 = load_dataset("CIIRC-NLP/czech_news_simple-cs", "default")

README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/649 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/503 [00:00<?, ?B/s]

data/train-00000-of-00001-93f855e5cee8b5(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001-8fd5c2a953da2a2(…):   0%|          | 0.00/2.67M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
print(type(ds01),'\n',ds01)
print(type(ds02),'\n',ds02)
print(type(ds03),'\n',ds03)
print(type(ds04),'\n',ds04)
print(type(ds05),'\n',ds05)
print(type(ds06),'\n',ds06)

<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    test: Dataset({
        features: ['type', 'query', 'choices', 'gold', 'context', 'subject', 'difficulty', 'source'],
        num_rows: 649
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['output', 'instruction'],
        num_rows: 4560
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 1641471
    })
    validation: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 144836
    })
    test: Dataset({
        features: ['url', 'au

### ***We display data files in polars***

In [6]:
df01 = pl.from_arrow(ds01["test"].data.table)
df01.sample(5)

type,query,choices,gold,context,subject,difficulty,source
str,str,list[str],i64,str,str,str,str
"""MC""","""Která z následujících možností…","[""A. Jde o prozaický text, rým přerývaný se vyskytuje nejen v poslední sloce."", ""B. Jde o prozaický text, rým přerývaný se vyskytuje pouze v poslední sloce."", … ""D. Jde o text psaný ve verších, rým přerývaný se vyskytuje pouze v poslední sloce.""]",2,"""<bold>(1)</bold> Kdyby to naše…","""český jazyk a literatura""","""maturitní zkouška""","""podzim 2021"""
"""MC""","""Které z následujících tvrzení …","[""A. Výchozí text popírá informaci, že ze všech čajovníkových plantáží pouze ty azorské leží v nadmořské výšce 150 metrů."", ""B. Výchozí text popírá informaci, že na světě existuje alespoň jedno místo, kde čajovníkové plantáže leží ve vyšší nadmořské výšce než na Azorách."", … ""D. Z výchozího textu vyplývá, že žádná z čajovníkových plantáží včetně těch azorských neleží v nadmořské výšce 150 metrů.""]",2,"""Portugalské souostroví Azory, …","""český jazyk a literatura""","""šestiletá gymnázia""","""1. řádný termín 2021"""
"""MC""","""Které z následujících tvrzení …","[""A. V textu se nevyskytuje žádný nespisovný tvar slova."", ""B. V textu je v nespisovném tvaru užito jak slovo řezaný, tak slovo rys."", … ""D. V textu je v nespisovném tvaru užito jak slovo pronikavý, tak slovo oko.""]",0,"""Do třídy vešel muž s ostře řez…","""český jazyk a literatura""","""čtyřleté obory a nástavbová st…","""3. řádný termín 2021"""
"""MC""","""Která z následujících možností…","[""A. komunikace rodičů s dětmi"", ""B. proces sebepoznávání"", … ""D. tvaroslovné chyby""]",2,"""<bold>(1)</bold> Malé dítě se …","""český jazyk a literatura""","""maturitní zkouška""","""podzim 2017"""
"""MC""","""Které z následujících tvrzení …","[""A. Tento verš není osmislabičný a rýmuje se s veršem, který je osmislabičný."", ""B. Tento osmislabičný verš se rýmuje s veršem, který je také osmislabičný."", … ""D. Tento verš není osmislabičný a rýmuje se s veršem, který také není osmislabičný.""]",1,"""TEXT 1 (1) Žil jednou v Čechác…","""český jazyk a literatura""","""čtyřleté obory a nástavbová st…","""1. řádný termín 2022"""


In [7]:
df02 = pl.from_arrow(ds03["train"].data.table)
df02.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.novinky.cz/bydleni…",[],"""Netradiční dřevěná přístavba o…","""Původně to byl obyčejný cihlov…",[],16,"""Jak architekti s jistou dávkou…",null,4,"""Bydlení""",[],0,2,2014-10-14 00:00:00
"""https://www.novinky.cz/zena/st…",[],"""Jak na záděry na rukou""","""Suchá a odrostlá kůžička u neh…",[],21,"""""Pravdou je, že se záděra neud…",null,4,"""Žena""",[],0,4,2012-07-12 00:00:00
"""https://www.novinky.cz/kultura…","[""Věra Míšková""]","""Nová DVD: Manželská rána, Hobi…","""Hobit: Šmakova dračí poušť vyc…",[],4,"""Hobit: Šmakova dračí poušť - p…",null,4,"""Kultura""",[2],2,2,2014-11-04 00:00:00
"""https://www.seznamzpravy.cz/cl…","[""Lucie Stuchlíková""]","""Hádka ČSSD v Praze: Ministr Lu…","""Ministr nedokázal prosadit kan…",[],2,"""Ministr zdravotnictví a dlouho…",null,1,"""Domácí""",[2],2,1,2017-04-03 00:00:00
"""https://kladensky.denik.cz/neh…","[""Daniela Řečínská""]","""Při nehodě dodávky a vlaku byl…","""Velvary - Neobvyklá dopravní n…","[""Vlak"", ""Havárie"", … ""Tísňové volání""]",0,"""Při havárii byl lehce zraněn s…",null,5,null,[2],2,3,2013-06-26 00:00:00


In [8]:
df03_train = pl.from_arrow(ds03["train"].data.table)
df03_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://prachaticky.denik.cz/h…","[""Pavel Kortus""]","""Jako první okusil led Bohumil …","""České Budějovice – Od středy u…","[""Bohumil jank"", ""Hc mountfield"", … ""Mountfield""]",3,"""Jankova tréninková píle je už …",null,5,"""SPORT""",[1],1,4,2012-07-26 00:00:00
"""https://www.idnes.cz/olomouc/z…","[""Petra Klimková""]","""Nemám se za co stydět, říká po…","""Drtivou porážku ČSSD v Olomouc…","[""Olomoucký kraj"", ""Čssd"", … ""Olomouc""]",2,"""Co říkáte na výsledek voleb?Já…",3,2,"""Kraje""",[2],2,5,2018-10-12 00:00:00
"""https://www.novinky.cz/koktejl…",[],"""Němec dodělal vysokou školu čt…","""Do studia aplikovaných věd na …","[""Německo"", ""Škola"", … ""Koktejl""]",6,"""“Kdybych se odstěhoval do Kolí…",null,4,"""Koktejl""",[],0,5,2012-10-19 00:00:00
"""https://zpravy.aktualne.cz/zah…","[""Pavel Vondra""]","""Gorbačov: Je na čase spojit ka…","""Poslední vůdce SSSR kritizuje …","[""Poslední vůdce"", ""Nejhorší verze"", … ""Vladimir putin""]",1,"""Moskva - Poslední vůdce Sověts…",null,3,"""Zahraničí""",[1],1,5,2009-03-06 00:00:00
"""https://www.irozhlas.cz/zpravy…","[""Milan Kocourek""]","""V noci pokračovaly nepokoje v …","""Severní Irsko zažívá nejhorší …","[""Severní irsko"", ""Nepokoje"", … ""Evropa""]",1,"""Při nepokojích v severním Belf…",null,6,"""Zprávy ze světa""",[1],1,3,2010-07-14 00:00:00


In [9]:
df03_validation = pl.from_arrow(ds03["validation"].data.table)
df03_validation.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://zpravy.aktualne.cz/zah…",[],"""Kdo byl na Tchaj-wanu, nesmí d…","""Čína zakáže vstup na své území…","[""Čína"", ""Tchaj-wan"", … ""Francie""]",1,"""Deník napsal, že Vystrčila pro…",null,3,"""Zahraničí""",[],0,4,2020-09-10 00:00:00
"""https://www.irozhlas.cz/sport/…",[],"""Kubalík znovu spasil Chicago v…","""Chicago zdolalo díky trefě Dom…","[""Hokej"", ""Nhl"", … ""Chicago blackhawks""]",3,"""Chicago postoupilo z dvanáctéh…",null,6,"""Sport""",[],0,6,2020-08-08 00:00:00
"""https://www.idnes.cz/hobby/dom…","[""Marek Burza""]","""Vepřový tomahawk je hvězdou le…","""Vepřový tomahawk steak je zají…","[""Vaření"", ""Grilování"", … ""Letní grilování - omáčky""]",0,"""Obecně lze říct, že je to stea…",8,2,"""Hobby""",[1],1,6,2020-05-30 00:00:00
"""https://pribramsky.denik.cz/zp…","[""Jakub Šťástka""]","""Dobříš pro své občany pořídí k…","""Obyvatelé Dobříše budou moci z…","[""Dobříš"", ""Kompostoviště"", ""Chotilsko""]",0,"""Ještě během letošního prosince…",null,5,"""ZPRÁVY""",[1],1,7,2020-12-13 00:00:00
"""https://magazin.aktualne.cz/ba…",[],"""Královnino letní sídlo trpí kv…","""Zaměstnanci zámku Balmoral, kt…","[""Koronavirus"", ""Zámek"", … ""Skotsko""]",0,"""Veřejné toalety ve Spojeném kr…",null,3,"""Magazín""",[],0,2,2020-06-30 00:00:00


In [10]:
df03_test = pl.from_arrow(ds03["test"].data.table)
df03_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://jablonecky.denik.cz/zl…","[""Jiří Louda""]","""Muži vyhrožovali Okamurovi pob…","""Na začátku září se na sociální…","[""Tomio okamura"", ""Policie"", … ""Jablonec""]",0,"""„Po prošetření jsme ve zkrácen…",null,5,null,[1],1,3,2021-09-22 00:00:00
"""https://www.idnes.cz/sport/vol…","[""Petr Lundák""]","""Konec sezony? Mach vyhlíží pře…","""V týmové teplákovce pozorně sl…","[""Jihočeský kraj"", ""Radek mach"", … ""Extraliga mužů""]",0,"""V zápase s Karlovarskem si 27.…",0,2,"""Ostatní""",[1],1,1,2021-03-15 00:00:00
"""https://www.denik.cz/regiony/r…","[""Petr Kinšt""]","""Rok po ničivé bouři ve Stebně:…","""Následky bouře, která se před …",[],2,"""Příkladem může být dům rodiny …",null,5,"""Regiony""",[1],1,7,2022-06-26 00:00:00
"""https://www.irozhlas.cz/sport/…",[],"""Krejčíková si vylepšila žebříč…","""Tenistka Barbora Krejčíková se…","[""Tenis"", ""Barbora krejčíková"", … ""Atp""]",3,"""Krejčíková si minulý týden zah…",null,6,"""Sport""",[],0,1,2021-03-15 00:00:00
"""https://zpravy.aktualne.cz/zah…",[],"""""To je Chuck Norris?"" Video z …","""""Chuck Norris, James Bond, Joh…","[""Chuck norris"", ""Pretoria"", … ""Toyota land cruiser""]",1,"""Leo Prinsloo měl podle dvou rů…",null,3,"""Zahraničí""",[],0,1,2021-05-03 00:00:00


In [11]:
df04_train = pl.from_arrow(ds04["train"].data.table)
df04_train.sample(5)

sentence,choices,answer_idx
str,list[str],i64
"""____ se na postupu a oficiální…","[""Domluvila"", ""Domluvilo"", … ""Domluvil""]",2
"""Pondělí ____ očekávání meteoro…","[""splnila"", ""splnilo"", … ""splnil""]",1
"""Pestrý program složený z taneč…","[""byla"", ""bylo"", … ""byl""]",4
"""Je to kruté ale ____ jsem takt…","[""využila"", ""využilo"", … ""využil""]",4
"""Ten, kdo mě zná, by asi ____ k…","[""mohla"", ""mohlo"", … ""mohl""]",4


In [12]:
df04_test = pl.from_arrow(ds04["test"].data.table)
df04_test.sample(5)

sentence,choices,answer_idx
str,list[str],i64
"""Kamarádce zase v Iscare o žehl…","[""měla"", ""mělo"", … ""měl""]",0
"""Přijímání projektů se ____ dos…","[""uskutečnila"", ""uskutečnilo"", … ""uskutečnil""]",1
"""Spolu jsme také ____ dohromady…","[""dala"", ""dalo"", … ""dal""]",2
"""Své přátele ____ do pasti a ne…","[""zatáhla"", ""zatáhlo"", … ""zatáhl""]",4
"""____ se ho 22 dětí z Krouny, P…","[""Účastnila"", ""Účastnilo"", … ""Účastnil""]",1


In [13]:
df05_train = pl.from_arrow(ds05["train"].data.table)
df05_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.irozhlas.cz/zpravy…",[],"""Vojáci USA při cvičení v Bulha…","""Bulharský majitel továrničky n…","[""Usa"", ""Bulharsko"", … ""Výsadkář""]",1,"""Podle agentury řídila americká…",null,6,"""Zprávy ze světa""",[],0,4,2021-06-03 00:00:00,1,11
"""https://www.seznamzpravy.cz/cl…","[""Martin Čaban""]","""Vizita: Jak pošťouchnout k očk…","""Britský vědecký časopis Nature…","[""Newsletter vizita"", ""Zdravotnictví"", … ""Covid-19""]",2,"""Protože na stránkách asi nejpr…",null,1,"""Domácí""",[1],1,3,2022-06-08 00:00:00,2,131
"""https://www.idnes.cz/ekonomika…",[],"""Nejbohatší člověk na světě je …","""Šéf americké internetové spole…","[""Spojené státy americké"", ""Donald trump"", … ""Jeff bezos""]",5,"""„Podporujeme zaměření Bidenovy…",39.0,2,"""Ekonomika""",[],0,3,2021-04-07 00:00:00,7,32
"""https://zpravy.aktualne.cz/eko…","[""Kateřina Hovorková""]","""Hypotéka jako dědictví. Banky …","""Hypotéka na 40 i více let se m…","[""Hypoteční úvěr"", ""Úvěr"", … ""Česká bankovní asociace""]",5,"""„Vidíme, že mnohé naše klienty…",null,3,"""Ekonomika""",[2],2,5,2021-10-29 00:00:00,7,19
"""https://zpravy.aktualne.cz/eko…","[""Kateřina Hovorková""]","""Banky v Česku prochází revoluc…","""Po několika letech, kdy se o k…","[""Banka"", ""Česko"", … ""Equa bank""]",5,"""Rakouská bankovní skupina Raif…",null,3,"""Ekonomika""",[2],2,4,2021-04-29 00:00:00,7,141


In [14]:
df05_test = pl.from_arrow(ds05["test"].data.table)
df05_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://magazin.aktualne.cz/ku…","[""Boris Klepal""]","""Don Juan jako fantasmagorie kr…","""Plameny žhnoucí touhy, nedosti…","[""Opera"", ""Don juan"", … ""Josif stalin""]",4,"""Plameny představují zcela jino…",null,3,"""Klasická hudba""",[1],1,1,2022-06-13 00:00:00,4,132
"""https://www.irozhlas.cz/kultur…","[""Josef Kaňka""]","""Co vznikne, když spojíte Foo F…","""Jedenáct Cen Grammy, deset stu…","[""Foo fighters"", ""Dave grohl"", … ""Nové album""]",4,"""Hlavním tématem alba Hail Sati…",null,6,"""Kultura""",[1],1,5,2021-07-16 00:00:00,4,179
"""https://www.seznamzpravy.cz/cl…","[""Filip Harzer""]","""Slovenský guvernér uvázl v sít…","""Během dvou týdnů je to už třet…","[""Slovensko"", ""Korupce"", ""Zuzana čaputová""]",1,"""Prokurátor viní bývalého vysoc…",null,1,"""Svět""",[1],1,5,2021-10-15 00:00:00,1,112
"""https://www.novinky.cz/kultura…","[""Veronika Rodriguez""]","""Drama Bílá nemoc v době covidu…","""Na vlastní kůži zažil hrůzy vá…",[],4,"""Co se vám vybaví, když se řekn…",null,4,"""Kultura""",[2],2,6,2021-03-27 00:00:00,4,105
"""https://www.seznamzpravy.cz/cl…","[""Daniela Přádová""]","""„Cesta byla moc těžká.“ Ukraji…","""Mezi nejčastější pacienty nově…","[""Válka rusko-ukrajina"", ""Ukrajina"", … ""Pomoc ukrajině""]",2,"""Článek si můžete pustit také v…",null,1,"""Domácí""",[2],2,3,2022-03-23 00:00:00,2,10


In [15]:
df06 = pl.from_arrow(ds06["test"].data.table)
df06.sample(5)

url,headline,brief,keywords,category,content,category_unclean
str,str,str,list[str],i64,str,str
"""https://www.irozhlas.cz/ekonom…","""Ekonom o fúzi Avastu: Pro Česk…","""Největší obchod v českém byzny…","[""Avast"", ""Avast software"", … ""Ekonom""]",5,"""Ondřej Vlček fúzi zdůvodnil na…","""Ekonomika"""
"""https://www.seznamzpravy.cz/cl…","""Ve Valašských Kloboukách bude …","""Ve Valašských Kloboukách na Zl…","[""Valašské klobouky"", ""Holokaust""]",2,"""„Do Osvětimi a Terezína byli d…","""Regiony"""
"""https://www.seznamzpravy.cz/cl…","""V Ostopovicích postaví novou k…","""Současná knihovna dosluhuje a …","[""Česká republika"", ""Jihomoravský kraj"", … ""Stavby""]",2,"""Navíc bude nový sál sloužit za…","""Regiony"""
"""https://www.irozhlas.cz/zpravy…","""Po 24hodinovém výpadku v Liban…","""V Libanonu se podařilo obnovit…","[""Libanon"", ""Palivo"", … ""Blízký východ""]",1,"""Energetická síť v zemi přestal…","""Zprávy ze světa"""
"""https://www.idnes.cz/usti/zpra…","""Konec nočnímu trénování či děl…","""Nová sportovní hala zřejmě vyr…","[""Ústecký kraj"", ""Petr globočník"", … ""Sk bivoj litvínov""]",2,"""„Nová hala by měla sloužit pri…","""Kraje"""


### ***Creating a summary data file***

In [16]:
def build_unified_contexts(ds01, ds02, ds03, ds04, ds05, ds06):
    print("I am starting the unification of Czech text contexts using Polars...")
    
    # --- 1. News Corpus (Hynky v2 - 1.6M+ lines) ---
    # We glue the title, annotation and the entire body of the article into one massive block
    df_news = pl.from_arrow(ds03["train"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    
    # --- 2. Legal / Alpaca dataset (czech-justice-summ-alpaca-long) ---
    # Here it makes sense to combine the instruction and the answer into one continuous text
    df_alpaca = pl.from_arrow(ds02["train"].data.table).lazy().select([
        (pl.col("instruction") + "\n" + pl.col("output")).alias("context"),
        pl.lit("alpaca_justice").alias("source")
    ])
    
    # --- 3. CERMAT Dataset (cermat_czech_mc) ---
    # Here we have the 'context' column, which we can enrich with a question
    df_cermat = pl.from_arrow(ds01["test"].data.table).lazy().select([
        (pl.col("context") + "\nOtázka: " + pl.col("query")).alias("context"),
        pl.lit("cermat").alias("source")
    ])

    # --- 4. Concatenation of all streams ---
    # Polars performs vertical joins extremely fast without unnecessary duplicate memory allocation
    unified_lazy = pl.concat([df_news, df_alpaca, df_cermat])
    
    # --- 5. Text extraction and cleaning + Calculation of logical predicates for JLNN ---
    final_df = unified_lazy.filter(
        pl.col("context").is_not_null() & (pl.col("context").str.len_chars() > 50)
    ).select([
        # Clean resulting text (context)
        pl.col("context"),
        pl.col("source"),
        
        # [PREDICT A]: Detection of masculine nouns in the text (Axiom of subject-predicate agreement)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["chlapci", "muži", "dělníci", "studenti", "hráči", "učitelé", "lidi", "psi"]
        ).cast(pl.Float32).alias("has_masc_animate"),
        
        # [PREDICATE C]: Detection of subordinating conjunctions introducing subordinate clauses (Axiom of Punctuation)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["že", "protože", "aby", "který", "jelikož", "jakmile", "zatímco"]
        ).cast(pl.Float32).alias("has_subord_conjunction"),
        
        # Auxiliary check: Is there a comma present in the text? (Basis for evaluating the t-norm)
        pl.col("context").str.contains(",").cast(pl.Float32).alias("has_comma")
    ]).collect()
    
    print(f"Unification complete! Total number of entire contexts in dataset: {final_df.height:,}")
    return final_df

In [17]:
%%time
df_dataset = build_unified_contexts(ds01, ds02, ds03, ds04, ds05, ds06)
df_dataset.sample(5)

I am starting the unification of Czech text contexts using Polars...
Unification complete! Total number of entire contexts in dataset: 1,646,599
CPU times: user 1min 1s, sys: 27.3 s, total: 1min 28s
Wall time: 1min 23s


context,source,has_masc_animate,has_subord_conjunction,has_comma
str,str,f32,f32,f32
"""Zabil mačetou mladíka, trestán…","""news""",0.0,1.0,1.0
"""Na Starém bělidle vládla Babič…","""news""",0.0,1.0,1.0
"""Na oslavě 35. narozenin Copu z…","""news""",0.0,1.0,1.0
"""Fotbalové Brno teskní, na půl …","""news""",0.0,1.0,1.0
"""Vodafone od srpna zruší studen…","""news""",1.0,1.0,1.0


### ***Preparing the RAW dataset (Pure sentence extraction in Polars)***

In [18]:
def create_raw_sentence_dataset(unified_df):
    print("I extract clean sentence segments and classify their punctuation...")
    
    # This regex finds any characters that end with either a period, question mark, exclamation mark, comma,
    # or at the end of the entire text. This will keep the sign right at the end of the segment!
    regex_segment_with_punc = r"[^.\?!,]+[\.\?!,]?"

    raw_segments = (
        unified_df
        # 1. We extract all the matching sentence segments as a Word List
        .select(
            pl.col("context").str.extract_all(regex_segment_with_punc)
        )
        # 2. Expand the sheet into separate rows
        .explode("context")
        .rename({"context": "segment"})
        
        # 3. We clean up spaces at the beginning and end (the stuck punctuation will remain)
        .with_columns(pl.col("segment").str.strip_chars())
        
        # 4. FILTERING: We remove empty or too short things
        .filter(
            (pl.col("segment").str.len_chars() > 5) & 
            (pl.col("segment").str.len_chars() < 200) &
            pl.col("segment").str.contains(r"[a-zA-Zá-žÁ-Ž]")
        )
        
        # 5. CLASSIFICATION OF TERMINAL TYPE
        .with_columns([
            pl.when(pl.col("segment").str.ends_with("?")).then(pl.lit("otazník"))
            .when(pl.col("segment").str.ends_with("!")).then(pl.lit("vykřičník"))
            .when(pl.col("segment").str.ends_with(".")).then(pl.lit("tečka"))
            .when(pl.col("segment").str.ends_with(",")).then(pl.lit("čárka"))
            .otherwise(pl.lit("žádná"))
            .alias("punctuation_type")
        ])
        
        # 6. Random sample for our Ollama
        .sample(n=10000, with_replacement=False, seed=42)
    )
    
    print(f"Prepared {raw_segments.height} sentence segments for Ollama.")
    return raw_segments

In [19]:
%%time
raw_sentence_df = create_raw_sentence_dataset(df_dataset)
raw_sentence_df.head(5)

I extract clean sentence segments and classify their punctuation...
Prepared 10000 sentence segments for Ollama.
CPU times: user 59.1 s, sys: 14.4 s, total: 1min 13s
Wall time: 1min 12s


segment,punctuation_type
str,str
"""Pokud se to stává opakovaně,""","""čárka"""
"""Připomeňte si letošní změny: R…","""čárka"""
"""Ani to však značku patřící pod…","""tečka"""
"""dopadly do moře Severní Korea …","""čárka"""
"""„Někdy stačí i jen velká popel…","""čárka"""


### ***Export dataset***

In [20]:
def export_polars_to_pyarrow_dataset(polars_df, output_dir="processed_dataset"):
    print("Converting Polars DataFrame to PyArrow Table...")
    
    # We define a dictionary for translating Czech values ​​into English folder names
    translation_dict = {
        "otazník": "question_mark",
        "vykřičník": "exclamation_mark",
        "tečka": "period",
        "čárka": "comma",
        "žadná": "none"
    }
    
    # Temporarily add a column for folders in Polars and immediately convert to Arrow
    arrow_table = (
        polars_df
        .with_columns(
            pl.col("punctuation_type").replace(translation_dict).alias("folder_name")
        )
        .to_arrow()
    )
    
    print(f"PyArrow table ready. Number of rows: {arrow_table.num_rows:,}")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Saving PyArrow Dataset to folder '{output_dir}' with English names...")
    
    # Writing to disk - partition will be done according to "folder_name" (English)
    ds.write_dataset(
        data=arrow_table,
        base_dir=output_dir,
        format="parquet",
        partitioning=["folder_name"],
        use_threads=True,
        existing_data_behavior="overwrite_or_ignore"
    )
    
    print("Export completed successfully!")

In [21]:
%%time
export_polars_to_pyarrow_dataset(raw_sentence_df, output_dir="raw_segments_dataset")

Converting Polars DataFrame to PyArrow Table...
PyArrow table ready. Number of rows: 10,000
Saving PyArrow Dataset to folder 'raw_segments_dataset' with English names...
Export completed successfully!
CPU times: user 28.3 ms, sys: 16.6 ms, total: 44.9 ms
Wall time: 64.7 ms
